# SentinelAI: Stage 05 — Feature Engineering & Dual-Benchmark Data Splitting

**Target Corpus**: [`ml/data/processed/sentinel_1M_benchmark.parquet`](../data/processed/sentinel_1M_benchmark.parquet) ($1,000,000$ representative flows)  
**Master Corpus**: [`ml/data/processed/merged_sentinel_core.parquet`](../data/processed/merged_sentinel_core.parquet) ($20,432,819$ flows)  
**Preceding Reports**:  
* [`docs/stages/04_exploratory_data_analysis/05_data_quality_and_standardization_report.md`](../../docs/stages/04_exploratory_data_analysis/05_data_quality_and_standardization_report.md)  
* [`docs/stages/04_exploratory_data_analysis/06_exploratory_data_analysis_report.md`](../../docs/stages/04_exploratory_data_analysis/06_exploratory_data_analysis_report.md)  
**Outputs Produced**:  
* `ml/saved_models/feature_scaler.joblib` (Leak-free RobustScaler pipeline)  
* `ml/saved_models/class_weights.json` (Inverse class frequency weighting matrix)  
* `ml/data/processed/benchmark_splits_manifest.json` (Partition indices & metadata)  

---

## 🎯 Objectives of Stage 05

Building directly on the discoveries from **Stage 04 Exploratory Data Analysis & Quality Audit**, this notebook implements four mission-critical engineering pillars:

1. **Derived Domain Feature Engineering**:
   * Directional payload asymmetry: $\text{pkt\_len\_ratio} = \frac{\text{fwd\_pkt\_len\_mean}}{\text{bwd\_pkt\_len\_mean} + 10^{-6}}$
   * Mean byte density: $\text{bytes\_per\_packet} = \frac{\text{fwd\_bytes} + \text{bwd\_bytes}}{\text{fwd\_packets} + \text{bwd\_packets}}$

2. **Non-Linear Rate & Volume Compression**:
   * Apply $\log_e(1 + x)$ (`np.log1p`) across all heavy-tailed features to neutralize the **$508,941\times$ rate skewness** caused by the *microsecond denominator effect*.

3. **Dual-Benchmark Splitting Architecture**:
   * **Benchmark A (In-Distribution Stratified Split)**: $70\%$ Train, $15\%$ Validation, $15\%$ Test, preserving relative shares of all 8 threat families and 3 dataset sources.
   * **Benchmark B (Zero-Shot Cross-Dataset Benchmark)**: Train on **CIC-IDS2017 + CSE-CIC-IDS2018** $\rightarrow$ Zero-Shot evaluate on **UNSW-NB15** to test domain transferability and measure AWS memorization.

4. **Leak-Free Scaling & Class Weight Serialization**:
   * Fit `RobustScaler` **strictly on the Train partition** (zero leakage into Val/Test).
   * Compute inverse class frequency weights ($w_j = \frac{N}{K \cdot n_j}$) to protect rare Web Exploitation ($19.6\text{k}$) from volumetric flood gradient starvation.

In [1]:
import os
import sys
import json
import time
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import pyarrow as pa
import joblib
from sklearn.preprocessing import RobustScaler
from sklearn.model_selection import train_test_split

np.random.seed(42)
print("Environment initialized successfully.")

Environment initialized successfully.


## 1. Load the 1-Million Fast-Iterate Benchmark Corpus

We load `ml/data/processed/sentinel_1M_benchmark.parquet`. This dataset preserves **$100\%$ of all rare attack classes** (Web Exploitation, Reconnaissance, Fuzzing, Infiltration) while downsampling redundant background normal flows.

In [2]:
benchmark_path = "../data/processed/sentinel_1M_benchmark.parquet"
if not os.path.exists(benchmark_path):
    raise FileNotFoundError(f"{benchmark_path} not found. Run benchmark extraction first.")

t0 = time.time()
df_raw = pq.read_table(benchmark_path).to_pandas()
load_time = time.time() - t0

print(f"Loaded {len(df_raw):,} rows x {df_raw.shape[1]} columns in {load_time:.2f} seconds.")
print(f"Memory footprint: {df_raw.memory_usage().sum() / 1024**2:.2f} MB")
print(f"Null count: {df_raw.isnull().sum().sum()}")

print("\n--- Canonical Threat Family Distribution in Benchmark ---")
counts = df_raw['attack_family'].value_counts()
shares = df_raw['attack_family'].value_counts(normalize=True) * 100
summary_df = pd.DataFrame({'Flows': counts, 'Share (%)': shares.round(2)})
print(summary_df.to_string())

Loaded 1,000,000 rows x 19 columns in 0.32 seconds.
Memory footprint: 77.25 MB
Null count: 0

--- Canonical Threat Family Distribution in Benchmark ---


                     Flows  Share (%)
attack_family                        
Normal              415132      41.51
Reconnaissance      164782      16.48
Infiltration        162865      16.29
DDoS                 50000       5.00
Botnet               50000       5.00
Brute Force          50000       5.00
DoS                  50000       5.00
Other / Fuzzing      37535       3.75
Web & Exploitation   19686       1.97


## 2. Derived Feature Engineering

Based on the physical network behaviors observed during EDA:
1. **`pkt_len_ratio`**: Measures directional asymmetry in packet length:  
   $$\text{pkt\_len\_ratio} = \frac{\text{fwd\_pkt\_len\_mean}}{\text{bwd\_pkt\_len\_mean} + 10^{-6}}$$
   * Attackers exploiting web applications (SQL injection, XSS) send large forward payloads (high mean length) and receive compact HTTP error codes ($403$, $500$).

2. **`bytes_per_packet`**: Measures overall packet density:  
   $$\text{bytes\_per\_packet} = \frac{\text{fwd\_bytes} + \text{bwd\_bytes}}{\text{fwd\_packets} + \text{bwd\_packets}}$$
   * Distinguishes high-density media streaming from probe sweeps.

In [3]:
df_feat = df_raw.copy()

# 1. Directional Packet Length Ratio
df_feat['pkt_len_ratio'] = (df_feat['fwd_pkt_len_mean'] / (df_feat['bwd_pkt_len_mean'] + 1e-6)).astype(np.float32)

# 2. Mean Bytes per Packet
total_bytes = df_feat['fwd_bytes'] + df_feat['bwd_bytes']
total_pkts = (df_feat['fwd_packets'] + df_feat['bwd_packets']).clip(lower=1)
df_feat['bytes_per_packet'] = (total_bytes / total_pkts).astype(np.float32)

# Check for invalid values
assert not df_feat['pkt_len_ratio'].isnull().any(), "Nulls in pkt_len_ratio"
assert not df_feat['bytes_per_packet'].isnull().any(), "Nulls in bytes_per_packet"
assert not np.isinf(df_feat['pkt_len_ratio']).any(), "Infs in pkt_len_ratio"
assert not np.isinf(df_feat['bytes_per_packet']).any(), "Infs in bytes_per_packet"

print("Engineered 'pkt_len_ratio' and 'bytes_per_packet' successfully.")
print(f"pkt_len_ratio range: [{df_feat['pkt_len_ratio'].min():.4f}, {df_feat['pkt_len_ratio'].max():.4f}] (Median: {df_feat['pkt_len_ratio'].median():.4f})")
print(f"bytes_per_packet range: [{df_feat['bytes_per_packet'].min():.4f}, {df_feat['bytes_per_packet'].max():.4f}] (Median: {df_feat['bytes_per_packet'].median():.4f})")

Engineered 'pkt_len_ratio' and 'bytes_per_packet' successfully.
pkt_len_ratio range: [0.0000, 2920000000.0000] (Median: 0.3333)
bytes_per_packet range: [0.0000, 2595.5557] (Median: 41.1667)


## 3. Non-Linear Rate & Volume Log-Transformations (`np.log1p`)

Network flow rates and byte counts span up to $10^{12}$ due to the microsecond denominator effect. We apply $\log_e(1 + x)$ to all heavy-tailed features:
* `flow_bytes_per_sec`
* `flow_pkts_per_sec`
* `flow_duration_s`
* `flow_iat_mean_s`
* `fwd_iat_mean_s`
* `bwd_iat_mean_s`
* `fwd_bytes`
* `bwd_bytes`
* `fwd_packets`
* `bwd_packets`
* `fwd_pkt_len_mean`
* `bwd_pkt_len_mean`
* `pkt_len_ratio`
* `bytes_per_packet`

> **Crucial Rule**: Ratios already normalized in $[0, 1]$ (`traffic_symmetry_ratio`, `fwd_tcp_win_ratio`, `bwd_tcp_win_ratio`) and destination port (`dst_port`) **do not receive log transforms**.

In [4]:
log_transform_cols = [
    'flow_bytes_per_sec',
    'flow_pkts_per_sec',
    'flow_duration_s',
    'flow_iat_mean_s',
    'fwd_iat_mean_s',
    'bwd_iat_mean_s',
    'fwd_bytes',
    'bwd_bytes',
    'fwd_packets',
    'bwd_packets',
    'fwd_pkt_len_mean',
    'bwd_pkt_len_mean',
    'pkt_len_ratio',
    'bytes_per_packet'
]

skew_before = df_feat[log_transform_cols].skew()

for col in log_transform_cols:
    # Apply log1p safely (clipping lower at 0)
    df_feat[col] = np.log1p(df_feat[col].clip(lower=0)).astype(np.float32)

skew_after = df_feat[log_transform_cols].skew()

skew_comparison = pd.DataFrame({
    'Skew Before': skew_before.round(2),
    'Skew After log1p': skew_after.round(2),
    'Skew Reduction': (skew_before - skew_after).round(2)
})
print("--- Impact of np.log1p on Feature Skewness ---")
print(skew_comparison.to_string())

--- Impact of np.log1p on Feature Skewness ---
                    Skew Before  Skew After log1p  Skew Reduction
flow_bytes_per_sec       553.08             -0.18          553.26
flow_pkts_per_sec        528.83             -0.15          528.99
flow_duration_s            3.38              2.24            1.14
flow_iat_mean_s            7.30              3.80            3.51
fwd_iat_mean_s             6.81              3.28            3.52
bwd_iat_mean_s            14.38              4.19           10.19
fwd_bytes                110.89              0.26          110.62
bwd_bytes                156.91              0.49          156.43
fwd_packets              111.11              1.94          109.16
bwd_packets              154.57              1.49          153.08
fwd_pkt_len_mean           9.56             -0.06            9.62
bwd_pkt_len_mean           5.31              0.14            5.17
pkt_len_ratio             22.21              2.89           19.32
bytes_per_packet           4.

## 4. Dual-Benchmark Splitting Architecture

To eliminate the naive random splitting trap documented in Stage 04 (where $80\%$ of training and testing comes from CSE-CIC-IDS2018 AWS cloud traffic), we construct **two distinct evaluation benchmarks**:

### Benchmark A: In-Distribution Stratified Split (70 / 15 / 15)
* **Train ($70\%$)**: Model training partition ($700,000$ flows).
* **Validation ($15\%$)**: Early stopping and hyperparameter tuning ($150,000$ flows).
* **Test ($15\%$)**: Final in-distribution benchmark evaluation ($150,000$ flows).
* **Stratification**: Jointly stratified on `attack_family` to ensure every rare class is identically represented across Train, Val, and Test.

### Benchmark B: Zero-Shot Cross-Dataset Benchmark
* **Train Partition**: CIC-IDS2017 + CSE-CIC-IDS2018 ($907,156$ flows).
* **Test Partition**: UNSW-NB15 ($92,844$ flows).
* **Purpose**: Measures out-of-distribution transferability to unseen Australian enterprise network architecture.

In [5]:
# Prepare indices and stratification key
indices = np.arange(len(df_feat))
strat_key = df_feat['attack_family'].values

# 1. Benchmark A: In-Distribution Stratified 70/15/15 Split
train_idx, temp_idx = train_test_split(
    indices, test_size=0.30, random_state=42, stratify=strat_key
)

temp_strat = strat_key[temp_idx]
val_idx, test_idx = train_test_split(
    temp_idx, test_size=0.50, random_state=42, stratify=temp_strat
)

print(f"Benchmark A Partitions:")
print(f"  Train: {len(train_idx):,} flows ({len(train_idx)/len(df_feat)*100:.1f}%)")
print(f"  Val:   {len(val_idx):,} flows ({len(val_idx)/len(df_feat)*100:.1f}%)")
print(f"  Test:  {len(test_idx):,} flows ({len(test_idx)/len(df_feat)*100:.1f}%)")

# Verify stratification fidelity
train_dist = df_feat.iloc[train_idx]['attack_family'].value_counts(normalize=True)
test_dist = df_feat.iloc[test_idx]['attack_family'].value_counts(normalize=True)
strat_check = pd.DataFrame({'Train Share (%)': (train_dist*100).round(2), 'Test Share (%)': (test_dist*100).round(2)})
print("\n--- Stratification Consistency Check (Train vs Test) ---")
print(strat_check.to_string())

Benchmark A Partitions:
  Train: 700,000 flows (70.0%)
  Val:   150,000 flows (15.0%)
  Test:  150,000 flows (15.0%)



--- Stratification Consistency Check (Train vs Test) ---
                    Train Share (%)  Test Share (%)
attack_family                                      
Botnet                         5.00            5.00
Brute Force                    5.00            5.00
DDoS                           5.00            5.00
DoS                            5.00            5.00
Infiltration                  16.29           16.29
Normal                        41.51           41.51
Other / Fuzzing                3.75            3.75
Reconnaissance                16.48           16.48
Web & Exploitation             1.97            1.97


In [6]:
# 2. Benchmark B: Zero-Shot Cross-Dataset Partitions
mask_cic = df_feat['source_dataset'].isin(['CSE-CIC-IDS2018', 'CIC-IDS2017'])
mask_nb15 = df_feat['source_dataset'] == 'UNSW-NB15'

cross_train_idx = indices[mask_cic]
cross_test_idx = indices[mask_nb15]

print(f"Benchmark B (Cross-Dataset) Partitions:")
print(f"  Train (CIC-17 + CIC-18): {len(cross_train_idx):,} flows ({len(cross_train_idx)/len(df_feat)*100:.1f}%)")
print(f"  Test  (UNSW-NB15):       {len(cross_test_idx):,} flows ({len(cross_test_idx)/len(df_feat)*100:.1f}%)")

Benchmark B (Cross-Dataset) Partitions:
  Train (CIC-17 + CIC-18): 907,156 flows (90.7%)
  Test  (UNSW-NB15):       92,844 flows (9.3%)


## 5. Leak-Free Feature Scaling (`RobustScaler`)

We fit `RobustScaler` (median & interquartile range) **strictly on the Train partition (`train_idx`)**.
This guarantees that zero information from Validation or Test distributions leaks into the model.

### Feature Quarantine Rule
`source_dataset` is strictly isolated from model features.
Features passed to the scaler pipeline include all 18 physical flow and derived features.

In [7]:
feature_columns = [
    'dst_port',
    'flow_duration_s',
    'flow_iat_mean_s',
    'fwd_iat_mean_s',
    'bwd_iat_mean_s',
    'fwd_packets',
    'bwd_packets',
    'fwd_bytes',
    'bwd_bytes',
    'fwd_pkt_len_mean',
    'bwd_pkt_len_mean',
    'flow_bytes_per_sec',
    'flow_pkts_per_sec',
    'fwd_tcp_win_ratio',
    'bwd_tcp_win_ratio',
    'traffic_symmetry_ratio',
    'pkt_len_ratio',
    'bytes_per_packet'
]

print(f"Fitting RobustScaler on {len(feature_columns)} features strictly using Train slice ({len(train_idx):,} rows)...")
scaler = RobustScaler()
scaler.fit(df_feat.iloc[train_idx][feature_columns])

# Save scaler pipeline
os.makedirs("../saved_models", exist_ok=True)
scaler_path = "../saved_models/feature_scaler.joblib"
joblib.dump(scaler, scaler_path)
print(f"Serialized fitted scaler to {scaler_path}.")

Fitting RobustScaler on 18 features strictly using Train slice (700,000 rows)...


Serialized fitted scaler to ../saved_models/feature_scaler.joblib.


## 6. Inverse Class Frequency Weighting Matrix

In Stage 04, we identified that flood attacks (`DDoS` + `DoS`) outnumber Web Exploits by **$117:1$**.
Without sample weighting, gradient updates from volumetric floods drown out minority classes.

We calculate exact inverse class weights for balanced multi-class cross-entropy:
$$w_j = \frac{N}{K \cdot n_j}$$
where $N$ is total training flows, $K = 8$ classes, and $n_j$ is the frequency of class $j$.

In [8]:
train_families = df_feat.iloc[train_idx]['attack_family']
classes = np.unique(train_families)
N = len(train_families)
K = len(classes)

class_weights = {}
weight_table = []

for c in classes:
    n_j = (train_families == c).sum()
    w_j = float(N / (K * n_j))
    class_weights[c] = round(w_j, 4)
    weight_table.append({
        'Canonical Family': c,
        'Train Flows': n_j,
        'Inverse Weight': round(w_j, 4),
        'Relative Gradient Multiplier': f"{w_j / (N / (K * (train_families == 'Normal').sum())):.1f}x"
    })

weight_df = pd.DataFrame(weight_table).sort_values(by='Inverse Weight', ascending=False)
print("--- Inverse Class Frequency Weighting Matrix ---")
print(weight_df.to_string(index=False))

# Save class weights
weights_path = "../saved_models/class_weights.json"
with open(weights_path, 'w') as f:
    json.dump(class_weights, f, indent=2)
print(f"\nSaved class weights to {weights_path}.")

--- Inverse Class Frequency Weighting Matrix ---
  Canonical Family  Train Flows  Inverse Weight Relative Gradient Multiplier
Web & Exploitation        13780          5.6443                        21.1x
   Other / Fuzzing        26275          2.9601                        11.1x
            Botnet        35000          2.2222                         8.3x
       Brute Force        35000          2.2222                         8.3x
              DDoS        35000          2.2222                         8.3x
               DoS        35000          2.2222                         8.3x
      Infiltration       114006          0.6822                         2.5x
    Reconnaissance       115347          0.6743                         2.5x
            Normal       290592          0.2677                         1.0x

Saved class weights to ../saved_models/class_weights.json.


## 7. Serialize Benchmark Splitting Manifest

We serialize the partition indices into `ml/data/processed/benchmark_splits_manifest.json` so training and evaluation scripts can deterministically load the exact same data partitions.

In [9]:
manifest = {
    "benchmark_dataset": "sentinel_1M_benchmark.parquet",
    "total_flows": int(len(df_feat)),
    "feature_columns": feature_columns,
    "log_transformed_columns": log_transform_cols,
    "quarantined_columns": ["source_dataset"],
    "target_columns": ["label", "attack_family"],
    "benchmark_A_in_distribution": {
        "train_count": int(len(train_idx)),
        "val_count": int(len(val_idx)),
        "test_count": int(len(test_idx)),
        "train_indices": train_idx.tolist(),
        "val_indices": val_idx.tolist(),
        "test_indices": test_idx.tolist()
    },
    "benchmark_B_cross_dataset": {
        "train_count": int(len(cross_train_idx)),
        "test_count": int(len(cross_test_idx)),
        "train_indices": cross_train_idx.tolist(),
        "test_indices": cross_test_idx.tolist()
    }
}

manifest_path = "../data/processed/benchmark_splits_manifest.json"
with open(manifest_path, 'w') as f:
    json.dump(manifest, f)

manifest_size_mb = os.path.getsize(manifest_path) / (1024**2)
print(f"Saved complete splitting manifest to {manifest_path} ({manifest_size_mb:.2f} MB).")
print("Stage 05 Feature Engineering & Splitting successfully complete!")

Saved complete splitting manifest to ../data/processed/benchmark_splits_manifest.json (15.05 MB).
Stage 05 Feature Engineering & Splitting successfully complete!
